# Chapter 17 — Search the Reasoning Space

**Book alignment:** DSPy From First Principles, Chapter 17

**Question this notebook isolates:** When the search-time value function is satisfied by the first expansion, do Greedy / MCTS / Random Tree collapse into one measurement — and does every strategy then tie a single direct call, for both a weak and a strong model?

No LM is called here. The cells reproduce two mechanisms from the chapter (UCT selection, cache-collapsed branching) and then check the chapter's measured claims directly against the committed run artifacts under `.artifacts/dspy-book/ch17_reasoning_search/`.

In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
import sys
from collections import defaultdict

random.seed(0)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy


class TraceStep(dspy.Signature):
    """Produce the next reasoning step given what is known."""

    context: str = dspy.InputField()
    trace: str = dspy.InputField()
    reasoning: str = dspy.OutputField()


class ScoreReasoning(dspy.Signature):
    """Evaluate how promising a reasoning path is."""

    context: str = dspy.InputField()
    trace: str = dspy.InputField()
    score: str = dspy.OutputField()


reasoning_fn = dspy.Predict(TraceStep)
score_fn = dspy.Predict(ScoreReasoning)
print("dspy", dspy.__version__, "| generation and value signatures constructed, never executed")

In [ ]:
ART = REPO_ROOT / ".artifacts" / "dspy-book" / "ch17_reasoning_search"


def load_run(path: Path) -> dict:
    by_condition = defaultdict(list)
    for row in json.loads(path.read_text()):
        by_condition[row["condition"]].append(row)
    return by_condition


QWEN = load_run(ART / "results.json")                    # ollama_chat/qwen3:latest, 8B
MUSE = load_run(ART / "muse-go-full" / "results.json")   # muse-spark-1.3-contributor
FAIR = ["direct", "cot", "best_of_n", "greedy_tree", "mcts", "mcts_no_reflect", "random_tree"]
TREES = ["greedy_tree", "mcts", "mcts_no_reflect", "random_tree"]

print("reps per condition:", {k: len(v) for k, v in sorted(QWEN.items())})
print("qwen seeds:", sorted({r["seed"] for rows in QWEN.values() for r in rows}))
print("muse seeds:", sorted({r["seed"] for rows in MUSE.values() for r in rows}))

## Selection spends finite compute

UCT balances exploitation against exploration: an unvisited child must be tried first, and an under-explored child can outrank a higher-average leader. This is the policy the tree arms *would* have exercised had the search continued past simulation 1.

In [ ]:
def uct_value(visits, reward, parent_visits, ucb_weight=1.41):
    if visits == 0:
        return float("inf")
    exploitation = reward / max(1, visits)
    exploration = ucb_weight * math.sqrt(max(1e-9, math.log(max(1, parent_visits)) / visits))
    return exploitation + exploration


PARENT_VISITS = 13
children = [
    {"id": "unvisited", "visits": 0, "reward": 0.0},
    {"id": "leader", "visits": 10, "reward": 7.0},
    {"id": "underexplored", "visits": 3, "reward": 0.9},
]
for child in children:
    child["uct"] = uct_value(child["visits"], child["reward"], PARENT_VISITS)
    avg = child["reward"] / max(1, child["visits"])
    print(f"{child['id']:14s} avg={avg:.2f} uct={child['uct']:.4f}")
order = [c["id"] for c in sorted(children, key=lambda c: c["uct"], reverse=True)]
print("selection order:", order)

In [ ]:
by_id = {c["id"]: c for c in children}
assert order == ["unvisited", "underexplored", "leader"]
assert by_id["underexplored"]["uct"] > by_id["leader"]["uct"]
assert (by_id["underexplored"]["reward"] / 3) < (by_id["leader"]["reward"] / 10)
print("exploration term does real work: lower average outranks the leader")

## Cache collapses branching

Expansion asks for *another* candidate continuation from the same parent. A cache keyed on identical `(state, trace)` answers that request with the first continuation every time, so three children share one trace.

In [ ]:
class FakeGenerator:
    def __init__(self):
        self.calls = 0
        self.cache = {}

    def cached_expand(self, state, trace):
        key = (state, tuple(trace))
        if key in self.cache:
            return self.cache[key]
        self.calls += 1
        step = f"thought-{self.calls}"
        self.cache[key] = step
        return step

    def fresh_expand(self, state, trace):
        self.calls += 1
        return f"thought-{self.calls}"


gen = FakeGenerator()
parent_trace: list = []
cached_children = [gen.cached_expand("evidence", parent_trace) for _ in range(3)]
gen2 = FakeGenerator()
fresh_children = [gen2.fresh_expand("evidence", parent_trace) for _ in range(3)]
print("cached siblings:", cached_children, "unique:", len(set(cached_children)))
print("fresh siblings: ", fresh_children, "unique:", len(set(fresh_children)))

In [ ]:
assert len(set(cached_children)) == 1
assert len(set(fresh_children)) == 3
print("drawing a tree does not mean exploring one: report unique_traces, not calls")

## The fair trees never spent their budget

Section 17.8.2. In both runs, every fair tree arm — Greedy, MCTS, MCTS − reflection, Random — recorded the same shape: two nodes, depth 1, one unique trace, one generation call, zero reflection calls, three model calls total. The admissible scorer cleared the 0.85 early-stop threshold on the first expansion, so the twelve-simulation loop never ran. Best-of-N made all 24 of its calls.

In [ ]:
def shape(row):
    return (
        row["nodes_created"],
        row["max_depth_reached"],
        row["unique_traces"],
        row["reflection_calls"],
        row["total_lm_calls"],
    )


for label, run in (("qwen3", QWEN), ("muse", MUSE)):
    print(f"[{label}] (nodes, depth, unique_traces, reflection_calls, total_calls)")
    for cond in TREES:
        print(f"  {cond:16} {sorted({shape(r) for r in run[cond]})}")
    print(f"  {'best_of_n':16} total_calls={sorted({r['total_lm_calls'] for r in run['best_of_n']})}")

In [ ]:
for run in (QWEN, MUSE):
    for cond in TREES:
        for row in run[cond]:
            assert shape(row) == (2, 1, 1, 0, 3), (cond, row["rep"], shape(row))
    assert all(r["total_lm_calls"] == 24 for r in run["best_of_n"])
    assert all(r["total_lm_calls"] == 1 for r in run["direct"])
print("Greedy = MCTS = Random by measurement: the code paths that separate them never ran")
print("Best-of-N spent 8x a tree arm's calls (24 vs 3) for the same score")

## Every strategy ties the direct baseline — at two capability levels

Section 17.8. The comparison's headline is a null result: no fair strategy beat a single direct call. The weaker model sat at a 0.75 evaluator ceiling (filename literalism, not a wrong diagnosis); the stronger model cleared it to a flat 1.00. Within each model the spread across the seven fair strategies is at most one repetition's worth of noise.

In [ ]:
def mean_score(run, cond):
    return statistics.mean(r["final_diagnosis_score"] for r in run[cond])


for label, run in (("qwen3", QWEN), ("muse", MUSE)):
    means = {c: round(mean_score(run, c), 3) for c in FAIR}
    spread = round(max(means.values()) - min(means.values()), 3)
    print(f"[{label}] fair-condition mean final_diagnosis_score  (spread {spread}):")
    for cond, value in means.items():
        print(f"  {cond:16} {value}")
    print()

In [ ]:
qwen_direct = mean_score(QWEN, "direct")
muse_direct = mean_score(MUSE, "direct")

# Weaker model: no fair strategy beats the direct baseline by more than one flipped
# repetition (1/3), and the whole fair field sits inside a 0.1 band.
assert qwen_direct == 0.75
assert all(mean_score(QWEN, c) - qwen_direct <= 1 / 3 + 1e-9 for c in FAIR)
assert max(mean_score(QWEN, c) for c in FAIR) - min(mean_score(QWEN, c) for c in FAIR) < 0.1

# Stronger model: every fair strategy is an exact 1.00, the direct call included.
assert muse_direct == 1.0
assert all(mean_score(MUSE, c) == 1.0 for c in FAIR)

# The 0.75 -> 1.00 shift is the model, not the strategy.
assert muse_direct > qwen_direct
print("Question A (does extra compute help this task?) answered twice: no")
print("Question B (does tree search allocate a budget better?) unanswerable here: no budget was spent")

## The oracle arm is the only one that searched — and it is invalid by construction

Section 17.9. With early stopping disabled, `oracle_mcts` ran all 12 simulations, built 13 nodes, and reached depth 2–3 in both runs. Its search-time scorer reads the reference answer, so `valid_evidence` is `False`: the result measures the strength of the leak, not the quality of the reasoning. It is kept only as proof that the repaired machinery *can* explore when its value function does not saturate.

In [ ]:
for label, run in (("qwen3", QWEN), ("muse", MUSE)):
    oracle = run["oracle_mcts"]
    fair_depth = max(r["max_depth_reached"] for cond in TREES for r in run[cond])
    print(
        f"[{label}] oracle valid_evidence={sorted({r['valid_evidence'] for r in oracle})} "
        f"gen_calls={sorted({r['generation_calls'] for r in oracle})} "
        f"nodes={sorted({r['nodes_created'] for r in oracle})} "
        f"depth={sorted({r['max_depth_reached'] for r in oracle})} | fair-arm depth={fair_depth}"
    )

In [ ]:
for run in (QWEN, MUSE):
    oracle = run["oracle_mcts"]
    assert all(r["valid_evidence"] is False for r in oracle)
    assert all(r["generation_calls"] == 12 for r in oracle)
    assert all(r["nodes_created"] == 13 for r in oracle)
    assert min(r["max_depth_reached"] for r in oracle) >= 2
    assert max(r["max_depth_reached"] for r in oracle) == 3
    fair_depth = max(r["max_depth_reached"] for cond in TREES for r in run[cond])
    assert max(r["max_depth_reached"] for r in oracle) > fair_depth
print("the only arm that searched is the one whose result must be discarded on principle")

## What we earned

The seven-strategy comparison was run — seeds 17–19, three repetitions, on an 8B local model and a stronger hosted one — and it returned a null result: every strategy scored the same as a single direct call. It could not answer its second question at all, because the fair tree arms never spent their search budgets. Search does not know truth; search knows reward — and here the value function was satisfied before any search happened.

Notebook 18 / Chapter 18 closes that boundary across the whole optimization loop with an experimental firewall.